In [4]:
import nest_asyncio
import sys
from config import settings
from agent_client import LangGraphSDKClient

# Enable nested event loops for Jupyter compatibility
nest_asyncio.apply()

print(f"🔗 Target Render URL: {settings.RENDER_URL}")
print(f"🤖 Target Assistant ID: {settings.ASSISTANT_ID}")

# Instantiate the client
client = LangGraphSDKClient()
print("✅ Client instantiated successfully!")

🔗 Target Render URL: https://scout-research.onrender.com
🤖 Target Assistant ID: deep_research_agent
✅ Client instantiated successfully!


In [5]:
print("--- TEST 1: Thread Creation ---")

try:
    thread = await client.create_thread(metadata={"environment": "notebook_test"})
    thread_id = thread["thread_id"]
    print(f"✅ Created Thread ID: {thread_id}")
except Exception as e:
    print(f"❌ Thread Creation Failed: {e}")

--- TEST 1: Thread Creation ---
✅ Created Thread ID: 019fe20c-3e03-74d2-b90c-414a19a4f583


In [6]:
print(f"--- TEST 2: Streaming Run on Thread [{thread_id}] ---")

test_query = "Summarize recent developments in solid-state batteries in 2 lines."
input_payload = {"query": test_query}

try:
    print(f"🚀 Sending Query: '{test_query}'\n")
    
    async for chunk in client.stream_run(thread_id=thread_id, input_data=input_payload):
        event_type = chunk.event
        event_data = chunk.data

        if event_type == "updates":
            node_name = list(event_data.keys())[0]
            node_output = event_data[node_name]
            print(f"📍 Stage Change -> Node: [{node_name}]")
            
            # Check for interrupt requests (clarify_with_user)
            if isinstance(node_output, dict) and node_output.get("type") == "clarification_request":
                print(f"❓ Interrupt Detected! Prompt: {node_output.get('message')}")

        elif event_type == "messages":
            msg, meta = event_data
            if isinstance(msg, dict) and "content" in msg:
                print(msg["content"], end="", flush=True)

        elif event_type == "values":
            if "final_report" in event_data:
                print("\n\n✅ Stream Complete! Final report state captured.")

except Exception as e:
    print(f"\n❌ Streaming Error: {e}")

--- TEST 2: Streaming Run on Thread [019fe20c-3e03-74d2-b90c-414a19a4f583] ---
🚀 Sending Query: 'Summarize recent developments in solid-state batteries in 2 lines.'

📍 Stage Change -> Node: [clarify_with_user]
